# 项目 — 航空公司 AI 助手

现在我们把所学内容整合起来，为一家航空公司打造 AI 客户支持助手

In [ ]:
# （代码逻辑保持原样；以下为小白向中文旁注）
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# 【注】Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

# 【注】As an alternative, if you'd like to use Ollama instead of OpenAI
# 【注】Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# 【注】MODEL = "llama3.2"
# 【注】openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
# （代码逻辑保持原样；以下为小白向中文旁注）
response_list = []
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    response_list.append(response)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
response_list

In [ ]:
# 【注】response.choices[0].finish_reason=="tool_calls"
# 【注】response.choices[0].message.content
print(response_list[0])
print(response_list[0].choices)
print(response_list[0].choices[0])

## 工具（Tools）

工具是前沿 LLM 提供的一项极其强大的功能。

有了工具，你可以编写一个函数，并让 LLM 在响应过程中调用该函数。

听起来几乎有点诡异……我们是在给它在我们机器上运行代码的权力？

嗯，有那么一点。

In [ ]:
# 【注】Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


In [ ]:
get_ticket_price("London")

In [ ]:
# 【注】There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [ ]:
# 【注】And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]

In [ ]:
tools

## 让 OpenAI 使用我们的工具

要让 OpenAI「调用我们的工具」，有一些琐碎细节。

我们实际做的是：给 LLM 机会告知我们它希望我们运行该工具。

新的 chat 函数大致如下：

In [ ]:
# （代码逻辑保持原样；以下为小白向中文旁注）

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    print("Messages sent to model = ", messages)
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    print("Initial response = ", response)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        print("Tool call detected!, message = ", message)
        response = handle_tool_call(message)
        print("Tool response = ", response)
        messages.append(message)
        messages.append(response)
        print("New messages list = ", messages)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
        print("New response after tool call = ", response)
    
    return response.choices[0].message.content

In [ ]:
# 【注】We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## 让我们做几项改进

在一次响应中处理多个工具调用

一个接一个地处理多个工具调用

In [ ]:
# （代码逻辑保持原样；以下为小白向中文旁注）
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    print("Messages sent to model = ", messages)
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    print("Initial response = ", response)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        print("Tool call detected!, message = ", message)
        responses = handle_tool_calls(message)
        print("Tool response = ", responses)
        messages.append(message)
        messages.extend(responses)
        print("New messages list = ", messages)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
        print("New response after tool call = ", response)
    
    return response.choices[0].message.content

In [ ]:
# （代码逻辑保持原样；以下为小白向中文旁注）
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
# （代码逻辑保持原样；以下为小白向中文旁注）
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    print("Messages sent to model = ", messages)
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    print("Initial response = ", response)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        print("Tool call detected!, message = ", message)
        responses = handle_tool_calls(message)
        print("Tool response = ", responses)
        messages.append(message)
        messages.extend(responses)
        print("New messages list = ", messages)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        print("New response after tool call = ", response)
    
    return response.choices[0].message.content

In [ ]:
# （代码逻辑保持原样；以下为小白向中文旁注）
import sqlite3


In [ ]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [ ]:
# （代码逻辑保持原样；以下为小白向中文旁注）
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [ ]:
get_ticket_price("London")

In [ ]:
# （代码逻辑保持原样；以下为小白向中文旁注）
def set_ticket_price(city, price):
    print(f"DATABASE TOOL CALLED: Setting price for {city} to {price}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [ ]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [ ]:
get_ticket_price("Tokyo")

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## 练习

添加一个用于设置机票价格的工具！

In [ ]:
price_get_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

price_set_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
            "price": {
                "type": "number",
                "description": "The price of the ticket",
            }
        },
        "required": ["destination_city","price"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": price_get_function}, {"type": "function", "function": price_set_function}]

In [ ]:
# （代码逻辑保持原样；以下为小白向中文旁注）
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    print("Messages sent to model = ", messages)
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    print("Initial response = ", response)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        print("Tool call detected!, message = ", message)
        responses = handle_tool_calls(message)
        print("Tool response = ", responses)
        messages.append(message)
        messages.extend(responses)
        print("New messages list = ", messages)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        print("New response after tool call = ", response)
    
    return response.choices[0].message.content

In [ ]:
# （代码逻辑保持原样；以下为小白向中文旁注）
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        
        if tool_call.function.name == "set_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price = arguments.get('price')
            set_ticket_price(city, price)
            responses.append({
                "role": "tool",
                "content": f"Price for {city} set to {price}",
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">商业应用</h2>
            <span style="color:#181;">这几乎不必多说！你现在已能给 LLM 赋予行动能力。这个航空助手不再只会回答问题——它还可以与预订 API 交互来完成预订！</span>
        </td>
    </tr>
</table>